In [15]:
#imports
import ir_datasets
import numpy as np
from rank_bm25 import BM25Okapi
from tqdm import tqdm
import math
import random
import pickle

In [8]:
#seeds
random.seed(42)
np.random.seed(42)

In [9]:
def load_queries_and_qrels(split = "train"):
    #this part will load queries from ir_datasets
    ds = ir_datasets.load(f"trec-tot/2023/{split}")
    queries = []
    for q in ds.queries_iter():
        queries.append({
            "query_id": q.query_id,
            "text": q.text })

    qrels ={}
    for qrel in ds.qrels_iter():
        if qrel.query_id not in qrels:
            qrels[qrel.query_id] = {}
        qrels[qrel.query_id][qrel.doc_id] = qrel.relevance

    return queries, qrels

def load_corpus():
    #this part will load corpus from ir_datasets, combines page_title and text for indexing

    ds = ir_datasets.load("trec-tot/2023")
    doc_ids = []
    corpus_texts = []

    print("corpus is loading")
    for doc in tqdm(ds.docs_iter()):
        doc_ids.append(doc.doc_id)
        corpus_texts.append(doc.page_title + ". " + doc.text)
    return doc_ids, corpus_texts

In [10]:
#tokenization
def tokenize(text):
    return text.lower().split()
#i used whitespace tokenizer

In [11]:
def build_bm25_index(corpus_texts, k1=1.5, b=0.75):
    #this part will build the bm25 index from the corpus
    print("tokenizing the corpus")
    tokenized = [tokenize(doc) for doc in tqdm(corpus_texts)]
    print("building bm25 index")
    return BM25Okapi(tokenized, k1=k1, b=b)

In [12]:
def retrieve(index, doc_ids, queries, top_k=100):
    #this part will run the bm25 retrieval for all the queries
    run = {}
    for q in tqdm(queries, desc="retrieving"):
        scores = index.get_scores(tokenize(q["text"]))
        top_indices = np.argsort(scores)[::-1][:top_k]
        run[q["query_id"]] = [(doc_ids[i], float(scores[i])) for i in top_indices]
    return run

In [13]:
#in this part i will compute nDCG@k for a single query
def ndcg_at_k(ranked_doc_ids, relevant_docs, k=10):
    dcg = 0.0
    for rank, doc_id in enumerate(ranked_doc_ids[:k], start=1):
        rel = relevant_docs.get(doc_id, 0)
        dcg += rel / math.log2(rank +1)
    ideal = sorted(relevant_docs.values(), reverse=True)[:k]
    idcg=sum(rel / math.log2(i+2) for i, rel in enumerate(ideal))

    return dcg / idcg if idcg > 0 else 0.0

def recall_at_k(ranked_doc_ids, relevant_docs, k=100):
#i will compute recall@k for a single query
    total_relevant = sum(1 for v in relevant_docs.values() if v > 0)
    if total_relevant == 0:
        return 0.0
    retrieved_relevant = sum(1 for doc_id in ranked_doc_ids[:k] if relevant_docs.get(doc_id, 0) > 0)
    return retrieved_relevant / total_relevant

def evaluate(run, qrels, k_ndcg=10, k_recall=100): #this part will evaluate the retrieval part
    ndcg_scores = []
    recall_scores = []

    for qid, results in run.items():
        relevant = qrels.get(qid, {})
        ranked_ids = [doc_id for doc_id, _ in results]
        ndcg_scores.append(ndcg_at_k(ranked_ids, relevant, k_ndcg))
        recall_scores.append(recall_at_k(ranked_ids, relevant, k_recall))

    return {
        f"nDCG@{k_ndcg}": np.mean(ndcg_scores),
        f"Recall@{k_recall}": np.mean(recall_scores),
        "_per_query_ndcg": ndcg_scores,
        "_per_query_recall": recall_scores,
    }

In [16]:
#main
if __name__ == "__main__":
    K1= 1.5
    B = 0.75

    print("loading train queries and qrels")
    train_queries, train_qrels = load_queries_and_qrels("train")
    print("loading dev queries and qrels")
    dev_queries, dev_qrels = load_queries_and_qrels("dev")
    all_queries = train_queries + dev_queries
    all_qrels = {**train_qrels, **dev_qrels}

    doc_ids, corpus_texts = load_corpus() #corpus
    print(f"corpus size: {len(doc_ids)}")

    run_combinations = [
            (1.5, 0.75), 
            (0.5, 0.25),
            (2.0, 0.75),
            (1.5, 0.5),
]

    for k1, b in run_combinations:
        print(f"\n=== k1={k1}, b={b} ===")
        index = build_bm25_index(corpus_texts, k1=k1, b=b)
        run = retrieve(index, doc_ids, all_queries, top_k=100)
        scores = evaluate(run, all_qrels)
        print(f"nDCG@10:    {scores['nDCG@10']:.4f}")
        print(f"Recall@100: {scores['Recall@100']:.4f}")
    
        wandb.login(key=os.environ.get("WANDB_API_KEY"))
        wandb.init(project="is584-tot-retrieval", entity="ozgukan",
                   name=f"bm25_k1_{k1}_b_{b}")
        wandb.log({
            "k1": k1,
            "b": b,
            "nDCG@10": scores["nDCG@10"],
            "Recall@100": scores["Recall@100"]
        })
        wandb.finish()
    
        with open(f"bm25_run_k1_{k1}_b_{b}.pkl", "wb") as f:
            pickle.dump(run, f)
        print("saved")
    
    print("\all combinations done")

loading train queries and qrels
loading dev queries and qrels
corpus is loading


231852it [00:16, 14442.06it/s]


corpus size: 231852

=== k1=1.5, b=0.75 ===
tokenizing the corpus


100%|██████████████████████████████████████████████████████████████████████████████| 231852/231852 [00:16<00:00, 13953.35it/s]


building bm25 index


...

building bm25 index


retrieving: 100%|█████████████████████████████████████████████████████████████████████████| 300/300 [1:17:39<00:00, 15.53s/it]


nDCG@10:    0.0313
Recall@100: 0.1100


Recall@100,▁
b,▁
k1,▁
nDCG@10,▁
Recall@100,0.11
b,0.5
k1,1.5
nDCG@10,0.03126


saved
ll combinations done


In [6]:
#we dont need to run below here they are for wandb

In [27]:
import pickle

with open("bm25_run_k1_1.5_b_0.75.pkl", "wb") as f:
    pickle.dump(run, f)

print("run saved") #i am saving it, ,so i can use it later

run saved


In [4]:
import os
wandb.login(key=os.environ.get("WANDB_API_KEY"))

True

In [5]:
#i am using wandb
wandb.init(project="is584-tot-retrieval", entity="ozgukan", name="bm25_k1_1.5_b_0.75")
wandb.log({
    "k1": 1.5,
    "b": 0.75,
    "nDCG@10": 0.0408,
    "Recall@100": 0.1100
})
wandb.finish()
print("logged!")

Recall@100,▁
b,▁
k1,▁
nDCG@10,▁
Recall@100,0.11
b,0.75
k1,1.5
nDCG@10,0.0408


logged!
